# Insurance - Serve Fraud Detection Model

Creates a Databricks Model Serving endpoint for the insurance fraud detection model.

Set the `ENVIRONMENT` widget (or the `ENVIRONMENT` env var) to control which configuration is used:

| Setting | Staging | Production |
|---|---|---|
| Endpoint suffix | `-staging` | `-prod` |
| Model alias | Challenger | Champion |
| Workload size | Small | Medium |
| Scale to zero | Enabled | Disabled |

**Steps:**
1. Resolve environment configuration
2. Promote the latest model version to the appropriate alias
3. Create (or update) the serving endpoint
4. Wait for the endpoint to reach `READY` state
5. Send a test request and display the prediction

In [ ]:
%run ../_setup

In [ ]:
import json
import time

import mlflow
import requests

In [ ]:
# ---------------------------------------------------------------------------
# Environment configuration
# ---------------------------------------------------------------------------
INDUSTRY = "insurance"
cfg = INDUSTRY_CONFIG[INDUSTRY]
MODEL_NAME = cfg["registered_model_name"]  # insurance_fraud_detection_model

ENV_PROFILES = {
    "staging": {
        "endpoint_suffix": "-staging",
        "model_alias": "Challenger",
        "workload_size": "Small",
        "scale_to_zero_enabled": True,
    },
    "production": {
        "endpoint_suffix": "-prod",
        "model_alias": "Champion",
        "workload_size": "Medium",
        "scale_to_zero_enabled": False,
    },
}

# Resolve environment: Databricks widget > env var > default to staging
try:
    ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT").lower()  # noqa: F821
except Exception:
    ENVIRONMENT = os.environ.get("ENVIRONMENT", "staging").lower()

if ENVIRONMENT not in ENV_PROFILES:
    raise ValueError(
        f"Unknown ENVIRONMENT '{ENVIRONMENT}'. Must be one of: {list(ENV_PROFILES.keys())}"
    )

env = ENV_PROFILES[ENVIRONMENT]
ENDPOINT_NAME = f"insurance-fraud-detection{env['endpoint_suffix']}"
MODEL_ALIAS = env["model_alias"]

print(f"Environment  : {ENVIRONMENT}")
print(f"Endpoint     : {ENDPOINT_NAME}")
print(f"Model alias  : {MODEL_ALIAS}")
print(f"Workload size: {env['workload_size']}")
print(f"Scale to zero: {env['scale_to_zero_enabled']}")

In [ ]:
# ---------------------------------------------------------------------------
# Promote the latest model version to the environment alias
# ---------------------------------------------------------------------------
client = mlflow.MlflowClient()

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if not versions:
    raise RuntimeError(f"No versions found for model '{MODEL_NAME}'. Run 02_train_model first.")

latest_version = max(versions, key=lambda v: int(v.version))
print(f"Model        : {MODEL_NAME}")
print(f"Version      : {latest_version.version}")
print(f"Run ID       : {latest_version.run_id}")

client.set_registered_model_alias(MODEL_NAME, MODEL_ALIAS, latest_version.version)
print(f"Alias set    : {MODEL_ALIAS} -> v{latest_version.version}")

In [ ]:
# ---------------------------------------------------------------------------
# Create or update the serving endpoint
# ---------------------------------------------------------------------------
WORKSPACE_HOST = mlflow.utils.databricks_utils.get_webapp_url()
TOKEN = mlflow.utils.databricks_utils.get_databricks_host_creds().token

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}

ENDPOINT_URL = f"{WORKSPACE_HOST}/api/2.0/serving-endpoints"

endpoint_config = {
    "name": ENDPOINT_NAME,
    "config": {
        "served_entities": [
            {
                "entity_name": MODEL_NAME,
                "entity_version": latest_version.version,
                "workload_size": env["workload_size"],
                "scale_to_zero_enabled": env["scale_to_zero_enabled"],
            }
        ]
    },
}

# Check whether the endpoint already exists
check_resp = requests.get(f"{ENDPOINT_URL}/{ENDPOINT_NAME}", headers=HEADERS)

if check_resp.status_code == 200:
    update_resp = requests.put(
        f"{ENDPOINT_URL}/{ENDPOINT_NAME}/config",
        headers=HEADERS,
        json=endpoint_config["config"],
    )
    update_resp.raise_for_status()
    print(f"Updated existing endpoint: {ENDPOINT_NAME}")
else:
    create_resp = requests.post(
        ENDPOINT_URL,
        headers=HEADERS,
        json=endpoint_config,
    )
    create_resp.raise_for_status()
    print(f"Created new endpoint: {ENDPOINT_NAME}")

print(f"Model        : {MODEL_NAME} v{latest_version.version}")
print(f"Workload size: {env['workload_size']}")
print(f"Scale to zero: {env['scale_to_zero_enabled']}")

In [ ]:
# ---------------------------------------------------------------------------
# Wait for the endpoint to be ready
# ---------------------------------------------------------------------------
POLL_INTERVAL_SECONDS = 30
TIMEOUT_SECONDS = 1200  # 20 minutes

elapsed = 0
while elapsed < TIMEOUT_SECONDS:
    resp = requests.get(f"{ENDPOINT_URL}/{ENDPOINT_NAME}", headers=HEADERS)
    resp.raise_for_status()
    status = resp.json()

    state = status.get("state", {})
    ready_state = state.get("ready", "UNKNOWN")
    config_update = state.get("config_update", "UNKNOWN")

    print(f"[{elapsed:>4d}s] ready={ready_state}  config_update={config_update}")

    if ready_state == "READY":
        print("\nEndpoint is READY.")
        break

    if ready_state == "FAILED" or config_update == "UPDATE_FAILED":
        raise RuntimeError(
            f"Endpoint entered a failed state: {json.dumps(state, indent=2)}"
        )

    time.sleep(POLL_INTERVAL_SECONDS)
    elapsed += POLL_INTERVAL_SECONDS
else:
    raise TimeoutError(
        f"Endpoint did not become ready within {TIMEOUT_SECONDS} seconds."
    )

In [ ]:
# ---------------------------------------------------------------------------
# Test the endpoint with a sample request
# ---------------------------------------------------------------------------
SCORING_URL = f"{WORKSPACE_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"

sample_payload = {
    "dataframe_records": [
        {
            "claim_amount": 85000,
            "policy_tenure_months": 36,
            "customer_age": 42,
            "num_prior_claims": 1,
            "premium_amount": 2400,
            "days_to_report": 3,
            "witness_present": 1,
            "police_report_filed": 1,
            "vehicle_age_years": 5,
            "incident_type": "Collision",
            "province": "Western Cape",
        }
    ]
}

score_resp = requests.post(SCORING_URL, headers=HEADERS, json=sample_payload)
score_resp.raise_for_status()

result = score_resp.json()
print("Sample request payload:")
print(json.dumps(sample_payload, indent=2))
print("\nEndpoint response:")
print(json.dumps(result, indent=2))

In [ ]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print(f"\n{'='*60}")
print(f"  Insurance Fraud Detection - Serving Endpoint")
print(f"{'='*60}")
print(f"  Environment   : {ENVIRONMENT}")
print(f"  Endpoint name : {ENDPOINT_NAME}")
print(f"  Model         : {MODEL_NAME} v{latest_version.version}")
print(f"  Alias         : {MODEL_ALIAS}")
print(f"  Workload size : {env['workload_size']}")
print(f"  Scale to zero : {env['scale_to_zero_enabled']}")
print(f"  Endpoint URL  : {SCORING_URL}")
print(f"  Workspace     : {WORKSPACE_HOST}")
print(f"{'='*60}")
print(f"\nView the endpoint in the Databricks UI:")
print(f"  {WORKSPACE_HOST}/#/mlflow/endpoints/{ENDPOINT_NAME}")